In [9]:
import pandas as pd
import json

In [10]:
# Load the batch response JSONL file and extract custom_id and response message content
response_answerable = []
with open('../batch/responses_answerable_20250914_142207.jsonl', 'r') as f:
    for line in f:
        data = json.loads(line)
        res = json.loads(data.get('response').replace('```json', '').replace('```', ''))
        response_answerable.append({
            'id': data.get('custom_id'),
            'llm_label': res['answerable'],
            'llm_reason': res['reason']
        })

response_df = pd.DataFrame(response_answerable)
response_df.head()

,id,llm_label,llm_reason
0,81e801d1-39cc-4d64-ab64-10619c171673,False,Question explicitly references options.
1,07cb00a8-27aa-4917-a9b9-369c38b87aac,False,Question explicitly references options with 'e...
2,65f2ce5a-dd89-45fc-a2cd-5f369acffe77,False,The question explicitly references options to ...
3,59165dbb-66fe-4ccd-bd9f-26c60ca3c161,False,Question explicitly references options with 't...
4,9d193952-cd22-4aab-a77e-c28dbfa697ea,False,Question explicitly references options.


In [11]:
# Load the batch response JSONL file and extract custom_id and response message content
batch_answerable = []
with open('../batch/batch_68c738e8ec10819091b7114e956e724e_output.jsonl', 'r') as f:
    for line in f:
        data = json.loads(line)
        print(data.get('response').get('body').get('choices')[0].get('message').get('content').replace('```json', '').replace('```', ''))
        res = json.loads(data.get('response').get('body').get('choices')[0].get('message').get('content').replace('```json', '').replace('```', ''))
        batch_answerable.append({
            'id': data.get('custom_id'),
            'llm_label': res['answerable'],
            'llm_reason': res['reason']
        })

batch_df = pd.DataFrame(batch_answerable)

{"answerable":false,"reason":"Question explicitly references options."}
{"answerable":false,"reason":"The question explicitly references options by asking for the 'most likely diagnosis' without providing them."}
{"answerable":false,"reason":"Question explicitly references options."}
{"answerable":false,"reason":"The question references a specific diagnosis with an implied list of options."}
{"answerable":false,"reason":"The question explicitly references options by asking to 'select the most likely diagnosis'."}
{"answerable":false,"reason":"Question explicitly references options."}
{"answerable":true,"reason":"The question is asking for a diagnosis based on symptoms and does not reference specific options."}
{"answerable":true,"reason":"The question describes symptoms and allows for a diagnosis based on the provided information."}
{"answerable":false,"reason":"Question explicitly references options with 'except'."}
{"answerable":false,"reason":"The question explicitly references opti

In [12]:
treatment_df = pd.read_csv('treatment.csv', index_col=0)

df = pd.merge(treatment_df, batch_df, on='id', how='left')
df.to_csv('treatment_with_answerable.csv')
df.head()

,id,question,opa,opb,opc,opd,cop,choice_type,exp,subject_name,topic_name,llm_label,llm_reason
0,1d4ccb9d-1924-4aa6-b07c-0ed46fe31c20,Which of the following is the best procedure d...,Fetal echocardiography,Fetal scalp pH,Continuous electrical fetal hea monitoring,Physical examination,2,single,Electrical Fetal hea monitoring is useful as: ...,Gynaecology & Obstetrics,"Intra Uterine Growth Restriction, Intrapaum an...",False,Question explicitly references options.
1,aeb420c2-ede8-48bc-9400-a5ae531f15ee,The radiograph of a 32 year old patient is sho...,Stafne’s bone cavity,Radicular Cyst,Dentigerous cyst,Lateral periodontal cyst,0,single,Radiological signs:\nThe lesion presents as a ...,Radiology,NaN,False,The question explicitly references options by ...
2,31868f6c-233a-40ee-880e-9c668509a8b1,An ill 16 days old baby girl is brought to the...,CHF,Glycogen storage disease,Pericarditis,Aberrant left coronary aery arising from pulmo...,0,multi,"In CHF pallor, dyspnoea, tachypnoea, tachycard...",Pediatrics,Impoant Viral Diseases in Children,False,Question explicitly references options.
3,62f6d5bb-6085-4328-97a0-3e3776f9ae78,A patient with cushinoid features presents wit...,Adrenal hyperplasia,Adrenal adenoma,Ca lung with ectopic ACTH production,Pituitary microadenoma,2,single,Answer is C (Ca lung with ectopic ACTH product...,Medicine,NaN,False,The question references a specific diagnosis w...
4,cbd91783-e901-4672-9ec1-7f58bce041da,A 74-year-old woman develops occipital headach...,basal ganglia hemorrhage,cerebellar hemorrhage,pontine hemorrhage,lobar intracerebral hemorrhage,1,multi,"Cerebellar hemorrhage, when mild, may present ...",Medicine,C.N.S.,False,The question explicitly references options by ...


In [13]:
df['llm_label'].value_counts()

llm_label
False    4235
True     3148
Name: count, dtype: int64